In [22]:
# Version 1.0 of this script uses Basic Pitch to transcribe audio and compare it to a target MIDI file, providing feedback on the accuracy of the performance.

from basic_pitch.inference import predict
import pretty_midi
from difflib import SequenceMatcher

def transcribe_audio(audio_path: str) -> pretty_midi.PrettyMIDI:
    """
    Runs Spotify's Basic Pitch on an audio file and returns PrettyMIDI.
    """
    model_output, midi_data, note_events = predict(
        audio_path,
        onset_threshold=0.5,       # Higher = fewer spurious note triggers
        frame_threshold=0.3,       # Confidence threshold for sustained notes
        minimum_note_length=58,    # Minimum note duration in ms (~58ms avoids pick scrape)
        minimum_frequency=80.0,    # ~E2 lowest guitar string in standard tuning
        maximum_frequency=1200.0   # Cutoff above typical guitar range
    )
    return midi_data

def extract_chord_sequence(midi_obj: pretty_midi.PrettyMIDI, onset_tolerance: float = 0.08) -> list[tuple[int, ...]]:
    """
    Extracts chronological pitch events. Simultaneous notes (dyads/chords)
    within `onset_tolerance` seconds are grouped into sorted tuples.
    Example output: [(64,), (60, 67), (62, 69)]
    """
    raw_notes = []
    for inst in midi_obj.instruments:
        if not inst.is_drum:
            raw_notes.extend(inst.notes)

    if not raw_notes:
        return []

    # Sort strictly by onset time, then by pitch
    raw_notes.sort(key=lambda n: (n.start, n.pitch))

    chord_sequence: list[tuple[int, ...]] = []
    current_cluster: list[int] = [raw_notes[0].pitch]
    cluster_anchor_time = raw_notes[0].start

    for note in raw_notes[1:]:
        # If this note starts within the tolerance window of the cluster anchor:
        if note.start - cluster_anchor_time <= onset_tolerance:
            if note.pitch not in current_cluster:
                current_cluster.append(note.pitch)
        else:
            chord_sequence.append(tuple(sorted(current_cluster)))
            current_cluster = [note.pitch]
            cluster_anchor_time = note.start

    if current_cluster:
        chord_sequence.append(tuple(sorted(current_cluster)))

    return chord_sequence

def pretty_chord_names(chord: tuple[int, ...]) -> str:
    return "+".join(pretty_midi.note_number_to_name(p) for p in chord)

def evaluate_guitar_riff(target_midi_path: str, student_audio_path: str):
    # 1. Parse target MIDI
    target_midi = pretty_midi.PrettyMIDI(target_midi_path)
    target_seq = extract_chord_sequence(target_midi, onset_tolerance=0.04)

    # 2. Transcribe student audio via Basic Pitch
    student_midi = transcribe_audio(student_audio_path)
    student_seq = extract_chord_sequence(student_midi, onset_tolerance=0.07)

    # 3. Sequence alignment
    matcher = SequenceMatcher(None, target_seq, student_seq)
    accuracy = matcher.ratio() * 100

    print(f"--- Riff Evaluation ---")
    print(f"Overall Accuracy: {accuracy:.1f}%\n")

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        target_chunk = [pretty_chord_names(c) for c in target_seq[i1:i2]]
        student_chunk = [pretty_chord_names(c) for c in student_seq[j1:j2]]

        if tag == 'equal':
            print(f"✅ Correct:      {', '.join(student_chunk)}")
        elif tag == 'replace':
            print(f"❌ Wrong Event:  Played [{', '.join(student_chunk)}], Expected [{', '.join(target_chunk)}]")
        elif tag == 'delete':
            print(f"⚠️  Missed:       Omitted [{', '.join(target_chunk)}]")
        elif tag == 'insert':
            print(f"⚠️  Extra Notes:  Added [{', '.join(student_chunk)}]")

In [23]:
# Version 1.1 of this script uses Basic Pitch to transcribe audio and compare it to a target MIDI file, providing feedback on the accuracy of the performance.

from basic_pitch.inference import predict
import pretty_midi
from difflib import SequenceMatcher

def transcribe_audio(audio_path: str) -> pretty_midi.PrettyMIDI:
    """
    Runs Spotify's Basic Pitch on an audio file with guitar-tuned thresholds.
    """
    model_output, midi_data, note_events = predict(
        audio_path,
        onset_threshold=0.6,       # Ignores pick scrapes / ghost taps
        frame_threshold=0.18,      # Catches quieter notes in dyads
        minimum_note_length=80,    # Eliminates short transient clicks
        minimum_frequency=80.0,    # Lowest guitar string (E2 ~82Hz)
        maximum_frequency=1200.0   # Ceiling above standard guitar range
    )
    return midi_data

def extract_chord_sequence(midi_obj: pretty_midi.PrettyMIDI, onset_tolerance: float = 0.08) -> list[tuple[int, ...]]:
    """
    Extracts chronological pitch events. Simultaneous notes (dyads/chords)
    within `onset_tolerance` seconds are grouped into sorted tuples.
    """
    raw_notes = []
    for inst in midi_obj.instruments:
        if not inst.is_drum:
            raw_notes.extend(inst.notes)

    if not raw_notes:
        return []

    # Sort chronologically by onset time, then by pitch
    raw_notes.sort(key=lambda n: (n.start, n.pitch))

    chord_sequence: list[tuple[int, ...]] = []
    current_cluster: list[int] = [raw_notes[0].pitch]
    cluster_anchor_time = raw_notes[0].start

    for note in raw_notes[1:]:
        if note.start - cluster_anchor_time <= onset_tolerance:
            if note.pitch not in current_cluster:
                current_cluster.append(note.pitch)
        else:
            chord_sequence.append(tuple(sorted(current_cluster)))
            current_cluster = [note.pitch]
            cluster_anchor_time = note.start

    if current_cluster:
        chord_sequence.append(tuple(sorted(current_cluster)))

    return chord_sequence

def pretty_chord_names(chord: tuple[int, ...]) -> str:
    """Converts a tuple of MIDI numbers like (40, 47) into 'E2+B2'."""
    return "+".join(pretty_midi.note_number_to_name(p) for p in chord)

def score_chord_match(target: tuple[int, ...], student: tuple[int, ...]) -> tuple[float, str]:
    """
    Fuzzy comparator evaluating guitar dyad accuracy:
    - 1.0: Exact note match
    - 0.85: Harmonic overtone shift (same note class, octave error)
    - 0.5-0.7: Partial chord (hit root, missed 5th or vice-versa)
    - 0.0: Complete miss
    """
    target_set = set(target)
    student_set = set(student)

    # 1. Exact match
    if target_set == student_set:
        return 1.0, "Spot on"

    # 2. Pitch-class match (octave / harmonic overtone shift: e.g., E3 instead of E2)
    target_classes = {p % 12 for p in target_set}
    student_classes = {p % 12 for p in student_set}

    if target_classes == student_classes:
        return 0.85, "Octave shift (acceptable overtone)"

    # 3. Partial chord match (hit some notes in the dyad/chord)
    common_pitches = target_classes.intersection(student_classes)
    if common_pitches:
        overlap_ratio = len(common_pitches) / len(target_classes)
        earned = 0.65 * overlap_ratio
        return earned, f"Partial dyad ({len(common_pitches)}/{len(target_classes)} notes hit)"

    return 0.0, "Wrong note"

def evaluate_guitar_riff(target_midi_path: str, student_audio_path: str):
    # 1. Parse target MIDI & transcribe student audio
    target_midi = pretty_midi.PrettyMIDI(target_midi_path)
    target_seq = extract_chord_sequence(target_midi, onset_tolerance=0.04)

    student_midi = transcribe_audio(student_audio_path)
    student_seq = extract_chord_sequence(student_midi, onset_tolerance=0.08)

    if not target_seq:
        print("Target MIDI contains no playable note events.")
        return

    # 2. Align the two sequences
    matcher = SequenceMatcher(None, target_seq, student_seq)
    
    total_score = 0.0
    total_target_notes = len(target_seq)

    print("--- 🎸 Riff Performance Breakdown ---")

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        target_chunk = target_seq[i1:i2]
        student_chunk = student_seq[j1:j2]

        if tag == 'equal':
            for chord in target_chunk:
                total_score += 1.0
                print(f"✅ Hit:          {pretty_chord_names(chord)}")

        elif tag == 'replace':
            # Pairwise inspection of the replaced chunk
            min_len = min(len(target_chunk), len(student_chunk))
            for k in range(min_len):
                tgt = target_chunk[k]
                stu = student_chunk[k]
                score, feedback = score_chord_match(tgt, stu)
                total_score += score
                
                if score >= 0.8:
                    print(f"🟡 Close Enough: Played {pretty_chord_names(stu)} vs Expected {pretty_chord_names(tgt)} ({feedback})")
                elif score > 0.0:
                    print(f"🟠 Partial:      Played {pretty_chord_names(stu)} vs Expected {pretty_chord_names(tgt)} ({feedback})")
                else:
                    print(f"❌ Missed Note:  Played {pretty_chord_names(stu)} vs Expected {pretty_chord_names(tgt)}")

            # Account for length differences in replaced chunks
            if len(target_chunk) > min_len:
                for tgt in target_chunk[min_len:]:
                    print(f"⚠️  Omitted:      Skipped {pretty_chord_names(tgt)}")
            elif len(student_chunk) > min_len:
                for stu in student_chunk[min_len:]:
                    print(f"⚠️  Extra:        Added {pretty_chord_names(stu)}")

        elif tag == 'delete':
            for chord in target_chunk:
                print(f"⚠️  Omitted:      Skipped {pretty_chord_names(chord)}")

        elif tag == 'insert':
            for chord in student_chunk:
                print(f"⚠️  Extra:        Added {pretty_chord_names(chord)}")

    # 3. Calculate final grade
    final_percentage = max(0.0, min(100.0, (total_score / total_target_notes) * 100))
    print(f"\nFinal Score: {final_percentage:.1f}%")

In [26]:
#Version 1.2 of this script uses Basic Pitch to transcribe audio and compare it to a target MIDI file, providing feedback on the accuracy of the performance.
from basic_pitch.inference import predict
import pretty_midi

def transcribe_audio(audio_path: str) -> pretty_midi.PrettyMIDI:
    """
    Runs Spotify's Basic Pitch with balanced parameters to capture 
    dyads while rejecting transient clicks/string scrapes.
    """
    model_output, midi_data, note_events = predict(
        audio_path,
        onset_threshold=0.55,      # Higher threshold blocks false pick scrapes
        frame_threshold=0.25,      # Balanced: sensitive enough for quiet notes, not noisy
        minimum_note_length=70,    # Eliminates short transient clicks (<70ms)
        minimum_frequency=80.0,    # Lowest guitar note (~E2 = 82.4 Hz)
        maximum_frequency=1200.0   # Upper bound for typical guitar neck range
    )
    return midi_data

def extract_chord_sequence(midi_obj: pretty_midi.PrettyMIDI, onset_tolerance: float = 0.05) -> list[tuple[int, ...]]:
    """
    Extracts chronological pitch events. Simultaneous notes (dyads/chords)
    within `onset_tolerance` seconds are grouped into sorted tuples.
    """
    raw_notes = []
    for inst in midi_obj.instruments:
        if not inst.is_drum:
            raw_notes.extend(inst.notes)

    if not raw_notes:
        return []

    # Sort strictly by onset time, then pitch
    raw_notes.sort(key=lambda n: (n.start, n.pitch))

    chord_sequence: list[tuple[int, ...]] = []
    current_cluster: list[int] = [raw_notes[0].pitch]
    cluster_anchor_time = raw_notes[0].start

    for note in raw_notes[1:]:
        if note.start - cluster_anchor_time <= onset_tolerance:
            if note.pitch not in current_cluster:
                current_cluster.append(note.pitch)
        else:
            chord_sequence.append(tuple(sorted(current_cluster)))
            current_cluster = [note.pitch]
            cluster_anchor_time = note.start

    if current_cluster:
        chord_sequence.append(tuple(sorted(current_cluster)))

    return chord_sequence

def pretty_chord_names(chord: tuple[int, ...] | None) -> str:
    """Converts a tuple like (40, 47) into 'E2+B2'."""
    if chord is None:
        return "-"
    return "+".join(pretty_midi.note_number_to_name(p) for p in chord)

def score_chord_match(target: tuple[int, ...], student: tuple[int, ...]) -> tuple[float, str]:
    """
    Fuzzy comparator evaluating guitar performance accuracy:
    - 1.0: Exact note match
    - 0.85: Harmonic overtone shift (same note class, octave error)
    - 0.5-0.7: Partial chord (e.g., hit root, missed 5th in a power chord)
    - 0.0: Whiffed / completely wrong note
    """
    target_set = set(target)
    student_set = set(student)

    # 1. Exact match
    if target_set == student_set:
        return 1.0, "Spot on"

    # 2. Pitch-class check (octave invariance: e.g. E3 instead of E2)
    target_classes = {p % 12 for p in target_set}
    student_classes = {p % 12 for p in student_set}

    if target_classes == student_classes:
        return 0.85, "Acceptable octave/harmonic shift"

    # 3. Partial dyad/chord match
    common_pitches = target_classes.intersection(student_classes)
    if common_pitches:
        overlap_ratio = len(common_pitches) / len(target_classes)
        earned = 0.65 * overlap_ratio
        return earned, f"Partial dyad ({len(common_pitches)}/{len(target_classes)} notes hit)"

    return 0.0, "Wrong pitch"

def align_and_score_sequences(target_seq, student_seq, gap_penalty=-0.5):
    """
    Needleman-Wunsch dynamic alignment.
    Aligns musical events even when the student drops notes, adds pick scrapes,
    or plays out of phase.
    """
    n = len(target_seq)
    m = len(student_seq)

    # Initialize DP scoring matrix
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i * gap_penalty
    for j in range(m + 1):
        dp[0][j] = j * gap_penalty

    # Populate DP matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match_score, _ = score_chord_match(target_seq[i - 1], student_seq[j - 1])
            sim_score = (match_score * 2.5) - 1.0

            dp[i][j] = max(
                dp[i - 1][j - 1] + sim_score,  # Diagonal: aligned comparison
                dp[i - 1][j] + gap_penalty,    # Up: target note was missed
                dp[i][j - 1] + gap_penalty     # Left: student played extra note
            )

    # Backtrace path
    i, j = n, m
    aligned_pairs = []

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            match_score, feedback = score_chord_match(target_seq[i - 1], student_seq[j - 1])
            sim_score = (match_score * 2.5) - 1.0
            if round(dp[i][j], 4) == round(dp[i - 1][j - 1] + sim_score, 4):
                aligned_pairs.append(('match', target_seq[i - 1], student_seq[j - 1], match_score, feedback))
                i -= 1
                j -= 1
                continue

        if i > 0 and round(dp[i][j], 4) == round(dp[i - 1][j] + gap_penalty, 4):
            aligned_pairs.append(('missed', target_seq[i - 1], None, 0.0, "Omitted"))
            i -= 1
        else:
            aligned_pairs.append(('extra', None, student_seq[j - 1], 0.0, "Extra note"))
            j -= 1

    aligned_pairs.reverse()
    return aligned_pairs

def evaluate_guitar_riff(target_midi_path: str, student_audio_path: str):
    """
    Main evaluation workflow: loads target MIDI, runs Basic Pitch on audio,
    aligns via Needleman-Wunsch, and prints a note-by-note grading breakdown.
    """
    # 1. Parse target MIDI & transcribe student audio
    target_midi = pretty_midi.PrettyMIDI(target_midi_path)
    target_seq = extract_chord_sequence(target_midi, onset_tolerance=0.04)

    student_midi = transcribe_audio(student_audio_path)
    student_seq = extract_chord_sequence(student_midi, onset_tolerance=0.05)

    if not target_seq:
        print("Target MIDI contains no note events to compare against.")
        return

    # 2. Perform sequence alignment
    alignment = align_and_score_sequences(target_seq, student_seq)

    total_earned = 0.0
    print("--- 🎸 Riff Performance Breakdown ---")
    
    for action, tgt, stu, score, feedback in alignment:
        total_earned += score
        if action == 'match':
            tgt_str = pretty_chord_names(tgt)
            stu_str = pretty_chord_names(stu)
            if score >= 0.8:
                print(f"✅ Hit ({int(score*100)}%):    Played {stu_str:<12} Target: {tgt_str} ({feedback})")
            elif score > 0.0:
                print(f"🟠 Partial ({int(score*100)}%): Played {stu_str:<12} Target: {tgt_str} ({feedback})")
            else:
                print(f"❌ Wrong:       Played {stu_str:<12} Target: {tgt_str}")
        elif action == 'missed':
            print(f"⚠️  Missed:      (Silent)     Target: {pretty_chord_names(tgt)}")
        elif action == 'extra':
            print(f"⚠️  Extra Note:  Played {pretty_chord_names(stu):<12} Target: (None)")

    # 3. Final grade
    final_score = max(0.0, min(100.0, (total_earned / len(target_seq)) * 100))
    print(f"\nFinal Score: {final_score:.1f}%")

In [28]:
#Version 1.3 of this script uses Basic Pitch to transcribe audio and compare it to a target MIDI file, providing feedback on the accuracy of the performance.
from basic_pitch.inference import predict
import pretty_midi

def transcribe_audio(audio_path: str) -> pretty_midi.PrettyMIDI:
    """
    Runs Spotify's Basic Pitch with balanced parameters to capture 
    dyads while rejecting transient clicks/string scrapes.
    """
    model_output, midi_data, note_events = predict(
        audio_path,
        onset_threshold=0.55,      # Higher threshold blocks false pick scrapes
        frame_threshold=0.25,      # Balanced: sensitive enough for quiet notes, not noisy
        minimum_note_length=70,    # Eliminates short transient clicks (<70ms)
        minimum_frequency=80.0,    # Lowest guitar note (~E2 = 82.4 Hz)
        maximum_frequency=1200.0   # Upper bound for typical guitar neck range
    )
    return midi_data

def extract_chord_sequence(midi_obj: pretty_midi.PrettyMIDI, onset_tolerance: float = 0.05) -> list[tuple[int, ...]]:
    """
    Extracts chronological pitch events. Simultaneous notes (dyads/chords)
    within `onset_tolerance` seconds are grouped into sorted tuples.
    """
    raw_notes = []
    for inst in midi_obj.instruments:
        if not inst.is_drum:
            raw_notes.extend(inst.notes)

    if not raw_notes:
        return []

    # Sort strictly by onset time, then pitch
    raw_notes.sort(key=lambda n: (n.start, n.pitch))

    chord_sequence: list[tuple[int, ...]] = []
    current_cluster: list[int] = [raw_notes[0].pitch]
    cluster_anchor_time = raw_notes[0].start

    for note in raw_notes[1:]:
        if note.start - cluster_anchor_time <= onset_tolerance:
            if note.pitch not in current_cluster:
                current_cluster.append(note.pitch)
        else:
            chord_sequence.append(tuple(sorted(current_cluster)))
            current_cluster = [note.pitch]
            cluster_anchor_time = note.start

    if current_cluster:
        chord_sequence.append(tuple(sorted(current_cluster)))

    return chord_sequence

def pretty_chord_names(chord: tuple[int, ...] | None) -> str:
    """Converts a tuple like (40, 47) into 'E2+B2'."""
    if chord is None:
        return "-"
    return "+".join(pretty_midi.note_number_to_name(p) for p in chord)

def score_chord_match(target: tuple[int, ...], student: tuple[int, ...]) -> tuple[float, str]:
    target_set = set(target)
    student_set = set(student)

    # 1. Exact match
    if target_set == student_set:
        return 1.0, "Spot on"

    target_classes = {p % 12 for p in target_set}
    student_classes = {p % 12 for p in student_set}

    # 2. Superset / Harmonic Overtone check:
    # If the student played all target pitch classes, do not penalize them
    # for picking up extra overtones/harmonics (e.g. G3+G4+D5 when target was G3)
    if target_classes.issubset(student_classes):
        return 1.0, "Spot on (overtones absorbed)"

    # 3. Pure Octave shift
    if target_classes == student_classes:
        return 0.85, "Acceptable octave/harmonic shift"

    # 4. Partial dyad/chord match (e.g., played 1 of 2 notes in a Hendrix double-stop)
    common_classes = target_classes.intersection(student_classes)
    if common_classes:
        # Give fair baseline credit for the hit note
        overlap_ratio = len(common_classes) / len(target_classes)
        earned = 0.5 + (0.35 * overlap_ratio)  # 1/2 notes gets ~67%, 2/3 gets ~73%
        return earned, f"Partial dyad ({len(common_classes)}/{len(target_classes)} notes hit)"

    return 0.0, "Wrong pitch"

def align_and_score_sequences(target_seq, student_seq, gap_penalty=-0.5):
    """
    Needleman-Wunsch dynamic alignment.
    Aligns musical events even when the student drops notes, adds pick scrapes,
    or plays out of phase.
    """
    n = len(target_seq)
    m = len(student_seq)

    # Initialize DP scoring matrix
    dp = [[0.0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i * gap_penalty
    for j in range(m + 1):
        dp[0][j] = j * gap_penalty

    # Populate DP matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match_score, _ = score_chord_match(target_seq[i - 1], student_seq[j - 1])
            sim_score = (match_score * 2.5) - 1.0

            dp[i][j] = max(
                dp[i - 1][j - 1] + sim_score,  # Diagonal: aligned comparison
                dp[i - 1][j] + gap_penalty,    # Up: target note was missed
                dp[i][j - 1] + gap_penalty     # Left: student played extra note
            )

    # Backtrace path
    i, j = n, m
    aligned_pairs = []

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            match_score, feedback = score_chord_match(target_seq[i - 1], student_seq[j - 1])
            sim_score = (match_score * 2.5) - 1.0
            if round(dp[i][j], 4) == round(dp[i - 1][j - 1] + sim_score, 4):
                aligned_pairs.append(('match', target_seq[i - 1], student_seq[j - 1], match_score, feedback))
                i -= 1
                j -= 1
                continue

        if i > 0 and round(dp[i][j], 4) == round(dp[i - 1][j] + gap_penalty, 4):
            aligned_pairs.append(('missed', target_seq[i - 1], None, 0.0, "Omitted"))
            i -= 1
        else:
            aligned_pairs.append(('extra', None, student_seq[j - 1], 0.0, "Extra note"))
            j -= 1

    aligned_pairs.reverse()
    return aligned_pairs

def evaluate_guitar_riff(target_midi_path: str, student_audio_path: str):
    """
    Main evaluation workflow: loads target MIDI, runs Basic Pitch on audio,
    aligns via Needleman-Wunsch, and prints a note-by-note grading breakdown.
    """
    # 1. Parse target MIDI & transcribe student audio
    target_midi = pretty_midi.PrettyMIDI(target_midi_path)
    target_seq = extract_chord_sequence(target_midi, onset_tolerance=0.04)

    student_midi = transcribe_audio(student_audio_path)
    student_seq = extract_chord_sequence(student_midi, onset_tolerance=0.05)

    if not target_seq:
        print("Target MIDI contains no note events to compare against.")
        return

    # 2. Perform sequence alignment
    alignment = align_and_score_sequences(target_seq, student_seq)

    total_earned = 0.0
    print("--- 🎸 Riff Performance Breakdown ---")
    
    for action, tgt, stu, score, feedback in alignment:
        total_earned += score
        if action == 'match':
            tgt_str = pretty_chord_names(tgt)
            stu_str = pretty_chord_names(stu)
            if score >= 0.8:
                print(f"✅ Hit ({int(score*100)}%):    Played {stu_str:<12} Target: {tgt_str} ({feedback})")
            elif score > 0.0:
                print(f"🟠 Partial ({int(score*100)}%): Played {stu_str:<12} Target: {tgt_str} ({feedback})")
            else:
                print(f"❌ Wrong:       Played {stu_str:<12} Target: {tgt_str}")
        elif action == 'missed':
            print(f"⚠️  Missed:      (Silent)     Target: {pretty_chord_names(tgt)}")
        elif action == 'extra':
            print(f"⚠️  Extra Note:  Played {pretty_chord_names(stu):<12} Target: (None)")

    # 3. Final grade
    final_score = max(0.0, min(100.0, (total_earned / len(target_seq)) * 100))
    print(f"\nFinal Score: {final_score:.1f}%")

In [29]:
audio_file_path = '/workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp3'
video_file_path = '/workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp4'
midi_file_path = '/workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mid'

evaluate_guitar_riff(midi_file_path, audio_file_path)

Predicting MIDI for /workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp3...
--- 🎸 Riff Performance Breakdown ---
✅ Hit (100%):    Played D4+E4        Target: D4+E4 (Spot on)
✅ Hit (100%):    Played E4           Target: E4 (Spot on)
✅ Hit (100%):    Played D4+E4        Target: D4+E4 (Spot on)
⚠️  Extra Note:  Played A2           Target: (None)
✅ Hit (100%):    Played B3+D4        Target: B3 (Spot on (overtones absorbed))
✅ Hit (100%):    Played A3+A4        Target: A3 (Spot on (overtones absorbed))
⚠️  Extra Note:  Played E4           Target: (None)
✅ Hit (100%):    Played G3+G4+D5     Target: G3 (Spot on (overtones absorbed))
✅ Hit (100%):    Played E3           Target: E3 (Spot on)
🟠 Partial (67%): Played D4           Target: D4+E4 (Partial dyad (1/2 notes hit))
✅ Hit (100%):    Played E4           Target: E4 (Spot on)
⚠️  Extra Note:  Played A2           Target: (None)
🟠 Partial (67%): Played D4           Target: D4+E4 (Partial dyad (1/2 notes hit))
⚠️  Miss

In [15]:
audio_file_path = '/workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp3'
video_file_path = '/workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp4'
midi_file_path = '/workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mid'


# Just load and run directly—Python can read 777/root files without issue
from basic_pitch.inference import predict
import pretty_midi

# Run inference directly on your test file
model_output, midi_data, note_events = predict(audio_file_path)
print(f"Detected {len(note_events)} notes!")

Predicting MIDI for /workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp3...
Detected 24 notes!


In [19]:
import io
import pretty_midi
import soundfile as sf
from basic_pitch.inference import predict
from IPython.display import Audio, display

# 1. Paths to your files
audio_path = audio_file_path  # or .wav / .mp4
target_midi_path = midi_file_path

# 2. Run Basic Pitch
print("Transcribing audio with Basic Pitch...")
model_output, predicted_midi, note_events = predict(audio_path)

# 3. Synthesize both MIDI objects to audio waveforms using pure sine waves
fs = 22050
print("Synthesizing transcribed MIDI...")
predicted_audio_synth = predicted_midi.synthesize(fs=fs)

print("Synthesizing target MIDI...")
target_midi = pretty_midi.PrettyMIDI(target_midi_path)
target_audio_synth = target_midi.synthesize(fs=fs)

# 4. Display playback players directly in the notebook
print("\n--- 1. Original Recording ---")
display(Audio(audio_path))

print("--- 2. What Basic Pitch Heard (Transcribed MIDI) ---")
display(Audio(predicted_audio_synth, rate=fs))

print("--- 3. Ground Truth Reference MIDI ---")
display(Audio(target_audio_synth, rate=fs))

Transcribing audio with Basic Pitch...
Predicting MIDI for /workspaces/dl_workspace/projects/guitar_instructor/uploads/heyjoe_snippet.mp3...
Synthesizing transcribed MIDI...
Synthesizing target MIDI...

--- 1. Original Recording ---


--- 2. What Basic Pitch Heard (Transcribed MIDI) ---


--- 3. Ground Truth Reference MIDI ---
